### Подключим библиотеки

In [ ]:
!pip install -q sentence-transformers rank_bm25 faiss-cpu transformers torch chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

### Загрузим очищенные файлы

In [ ]:
import os
from pathlib import Path

clean_dir = Path("clean_docs")
documents = []
for txt_path in clean_dir.glob("*.txt"):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    documents.append({"id": txt_path.stem, "text": text})
print(f"Загружено {len(documents)} документов")

Загружено 56 документов


## Делим на чанки

Будем использовать рекурсивный сплиттер с чанками по 384 токенов с перекрытием 128 токенов. Дополнительно следим чтобы не было разрывов предложений и абзацев.

In [ ]:
def recursive_split(text, chunk_size=384, chunk_overlap=128, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", " ", ""]
    if len(text) <= chunk_size:
        return [text]
    # ищем подходящий разделитель в пределах chunk_size
    for sep in separators:
        split_pos = text.rfind(sep, 0, chunk_size + 1)
        if split_pos != -1:
            break
    else:
        split_pos = chunk_size
    chunk = text[:split_pos].strip()
    # рекурсивно обрабатываем остаток с перекрытием
    overlap_start = max(0, split_pos - chunk_overlap)
    rest = text[overlap_start:]
    return [chunk] + recursive_split(rest, chunk_size, chunk_overlap, separators)

Здесь для создания эмбеддингов используем многоязычную модель paraphrase-multilingual-MiniLM-L12-v2. Также реализуем гибридный поиск с использованием BM25.

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
import faiss

def recursive_split(text, chunk_size=384, chunk_overlap=128, separators=":;"):
    if separators==":;":
        separators = ["\n\n", "\n", " ", ""]
    chunks = []
    start_index = 0
    while start_index < len(text):
        potential_chunk_end = min(start_index + chunk_size, len(text))
        split_at = -1
        for sep in separators:
            temp_split_at = text.rfind(sep, start_index, potential_chunk_end)
            if temp_split_at != -1:
                split_at = temp_split_at
                break
        if split_at == -1:
            split_at = potential_chunk_end
        if split_at == start_index and start_index < len(text):
            split_at = min(start_index + 1, len(text))
        current_chunk = text[start_index:split_at].strip()
        if current_chunk:
            chunks.append(current_chunk)
        next_start_index = max(start_index + 1, split_at - chunk_overlap)
        start_index = next_start_index
    return chunks

# Инициализация данных и моделей
all_chunks = []
for doc in documents:
    split_chunks = recursive_split(doc["text"])
    for i, ch in enumerate(split_chunks):
        all_chunks.append({"id": f"{doc['id']}-{i}", "text": ch})

# dense модель эмбеддингов
dense_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
chunk_texts = [c["text"] for c in all_chunks]
dense_embeddings = dense_model.encode(chunk_texts, show_progress_bar=True)
dim = dense_embeddings.shape[1]
# индекс для косинусного сходства
index = faiss.IndexFlatIP(dim)
# нормализация для косинусного сходства
faiss.normalize_L2(dense_embeddings)
# добавление эмбеддингов в индекс
index.add(dense_embeddings)

# sparse BM25 модель
tokenized_chunks = [text.split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_search(query, top_k=5, alpha=0.6):
    # 1. Плотный поиск
    q_emb = dense_model.encode([query])
    faiss.normalize_L2(q_emb)
    dense_scores, dense_indices = index.search(q_emb, top_k)

    # Словарь для отображения индекса чанка на плотную оценку
    dense_results = {}
    for idx, score in zip(dense_indices[0], dense_scores[0]):
        if idx != -1 and idx < len(all_chunks):
            dense_results[int(idx)] = float(score)

    # 2. Поиск BM25
    bm25_query_scores = bm25.get_scores(query.split())
    max_bm25 = np.max(bm25_query_scores) if np.max(bm25_query_scores) > 0 else 1.0

    # Берём top_k индексов по BM25
    bm25_top_indices = np.argsort(bm25_query_scores)[-top_k:]

    # 3. Объединение уникальных кандидатов
    candidates = set(dense_results.keys()) | set(bm25_top_indices.tolist())

    final_scored = []
    for idx in candidates:
        if idx < 0 or idx >= len(all_chunks):
            continue

        d_score = dense_results.get(idx, 0.0)          # плотная оценка (0 если не найден)
        s_score = bm25_query_scores[idx] / max_bm25   # нормализованная BM25 оценка

        combo_score = alpha * d_score + (1 - alpha) * s_score
        final_scored.append((all_chunks[idx], combo_score))

    # Сортируем по комбинированной оценке и возвращаем top_k
    final_scored.sort(key=lambda x: x[1], reverse=True)
    return final_scored[:top_k]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/418 [00:00<?, ?it/s]

## Переранжирование

Чтобы повысить точность релевантности используем кросс-енкодер cross-endocer/ms-macro-MiniLM-L-6-v2. Он работает так, что обрабатывает пару значений запрос-чанк, и выдает рейтинг релевантности, по нему отбирается позже итоговый топ-5. Пробовал делать с топ-3, но сделал топ-5 для большей точности ответа модели.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

rerank_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rerank_tokenizer = AutoTokenizer.from_pretrained(rerank_model_name)
rerank_model = AutoModelForSequenceClassification.from_pretrained(rerank_model_name)
rerank_model.eval()

def rerank(query, chunks_with_scores, top_n=3):
    pairs = [(query, chunk["text"]) for chunk, _ in chunks_with_scores]
    inputs = rerank_tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():
        scores = rerank_model(**inputs).logits.squeeze().tolist()
    if isinstance(scores, float):
        scores = [scores]
    reranked = sorted(zip(chunks_with_scores, scores), key=lambda x: x[1], reverse=True)
    return [chunk for (chunk, _), _ in reranked[:top_n]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Ответ LLM

Будем в этот  раз использовать модель через Ollama, так что для начала интегрируем модель в этот ноутбук.

Установим и настроим окружение.


In [ ]:
import os
import subprocess
import time
from google.colab import drive

# Монтируем Google Диск
drive.mount('/content/drive')

# Устанавливаем систему сборки и zstd
#!apt-get update
#!apt-get install -y build-essential zstd

# Устанавливаем Ollama через официальный скрипт
!curl -fsSL https://ollama.com/install.sh | sh

# Проверяем, что Ollama установлен и доступен из командной строки
!ollama --version

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Попробуем тестово проинициализировать ollama, так каак она работает как сервер.

In [ ]:
# Запускаем сервер Ollama в фоновом режиме
subprocess.Popen(['ollama', 'serve'])

# Даем серверу несколько секунд на инициализацию
print("Запуск сервера Ollama...")
time.sleep(5)
print("Сервер Ollama запущен.")

Запуск сервера Ollama...
Сервер Ollama запущен.


### Загрузка модели

По умолчанию модели как и все файлы в сессионном хранилище хранятся только до окончания сессии, так что надо это исправить, поменяв путь для загрузки моделей на гугл диск.

Здесь же сразу напишем команду для подгрузки модели. Потенциально использоваться будет только llama3.2:3b, но для проверки первоначально также использовалась tinyllama, так как она значительно легче (почти в 3 раза чем llama3.2:3b).

In [ ]:
# Создаем папку для моделей на Google Диске
!mkdir -p /content/drive/MyDrive/ollama_models

# Указываем Ollama использовать эту папку для хранения
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'

!ollama pull tinyllama:latest # легкая модель, использовали для тестов


#!ollama pull llama3.2:1b
#!ollama pull llama3.2:3b  # Модель 3B для качества и скорости
# !ollama pull mistral:7b    # Более мощная модель, может быть медленной на T4 GPU

# Проверяем список моделей
!ollama list


NAME                ID              SIZE      MODIFIED               
tinyllama:latest    2644915ede35    637 MB    Less than a second ago    
llama3.2:3b         a80c4f17acd5    2.0 GB    6 minutes ago             


## Интеграция модели

Напишем функцию для отправки запросов к серверу Ollama. Используем способ отправки запросов через библиотеку requests. Мы будем посылать POST запрос к серверу, данные отправляем как JSON.

Также сразу пропишем системный промпт модели:
```
Перечисли все конкретные действия, упомянутые в контексте, включая меры безопасности и приложения и тому подобное.
    
    Контекст:
{context_text}

Вопрос: {query}

Ответ на русском языке, только факты из контекста:
```

Напишем функцию для работы с моделью, и все, можно будет к ней обращаться. Также пропишем проверку сервера, на случай если он например упал.

In [ ]:
import requests
import time

# посмотрим, поднят ли сервер
try:
    r = requests.get("http://localhost:11434/api/tags")
    if r.status_code == 200:
        print("Сервер Ollama активен")
    else:
        print(f"Статус сервера: {r.status_code}")
except Exception as e:
    print(f"Сервер недоступен: {e}")

# функция для полноценного RAG цикла
def rag_answer(query, model="llama3.2:3b"):
    initial_results = hybrid_search(query, top_k=8, alpha=0.6)
    best_chunks = rerank(query, initial_results, top_n=3)

    print("Найденные чанки (первые 300 символов каждого)")
    for i, chunk in enumerate(best_chunks):
      print(f"Чанк {i+1}:\n{chunk['text'][:300]}\n")

    if not best_chunks:
        return "Не найдено релевантных фрагментов."

    context_text = "\n\n".join([c["text"] for c in best_chunks])

    prompt = f"""Перечисли все конкретные действия, упомянутые в контексте, включая меры безопасности и приложения и тому подобное.

    Контекст:
{context_text}

Вопрос: {query}

Ответ на русском языке, только факты из контекста:"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=120
        )
        if response.status_code == 200:
            return response.json()["response"].strip()
        else:
            return f"Ошибка API: {response.status_code}"
    except Exception as e:
        return f"Ошибка: {e}"



# Пример работы с реальными данными
user_query = "Какие правила включает в себя политика в отношении обработки персональных данных в Озон Инвест?"
print(f"Вопрос: {user_query}")
print("-" * 30)
final_answer = rag_answer(user_query)
print(f"Ответ модели:\n{final_answer}")

Сервер Ollama активен
Вопрос: Какие правила включает в себя политика в отношении обработки персональных данных в Озон Инвест?
------------------------------
Найденные чанки (первые 300 символов каждого)
Чанк 1:
ийской Федерации, если иное прямо не указано в Политике. 2. Принципы обработки персональных данных 2.1. Оператором разработаны и введены в действие внутренние документы и локальные нормативные акты, устанавливающие порядок обработки и защиты персональных данных, которые обеспечивают соответствие тре

Чанк 2:
поручения на обработку данных (с правом дальнейшей передачи третьим лицам (субобработчикам) в целях исполнения Договора); 9.5.2. Клиент обязуется соблюдать конфиденциальность ПДн и обеспечивать безопасность ПДн при их обработке, а также обеспечивает принятие необходимых правовых, организационных и т

Чанк 3:
соответствии с ним нормативных правовых актов, настоящей политики, внутренних документов и локальных актов по вопросам обработки персональных данных. 8.3. По запросу упо

Ответ корректен и в целом соответствует предписанной структуре, но требует доработки. Попробуем использовать несколько другой промпт:

```
"""Ты — ассистент, отвечающий строго на основе предоставленного контекста.
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."

**Контекст:**
{context_text}

**Вопрос:** {query}

**Ответ:**"""
```


In [ ]:
import requests
import time

# посмотрим, поднят ли сервер
try:
    r = requests.get("http://localhost:11434/api/tags")
    if r.status_code == 200:
        print("Сервер Ollama активен")
    else:
        print(f"Статус сервера: {r.status_code}")
except Exception as e:
    print(f"Сервер недоступен: {e}")

# функция для полноценного RAG цикла
def rag_answer(query, model="llama3.2:3b"):
    initial_results = hybrid_search(query, top_k=8, alpha=0.6)
    best_chunks = rerank(query, initial_results, top_n=5)

    print("Найденные чанки (первые 300 символов каждого)")
    for i, chunk in enumerate(best_chunks):
      print(f"Чанк {i+1}:\n{chunk['text'][:300]}\n")

    if not best_chunks:
        return "Не найдено релевантных фрагментов."

    context_text = "\n\n".join([c["text"] for c in best_chunks])

    prompt = f"""Ты — ассистент, отвечающий на основе предоставленного контекста.
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."

**Контекст:**
{context_text}

**Вопрос:** {query}

**Ответ:**"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=120
        )
        if response.status_code == 200:
            return response.json()["response"].strip()
        else:
            return f"Ошибка API: {response.status_code}"
    except Exception as e:
        return f"Ошибка: {e}"



# 3. Пример работы с реальными данными
user_query = "Какие правила включает в себя политика в отношении обработки персональных данных в Озон Инвест?"
print(f"Вопрос: {user_query}")
print("-" * 30)
final_answer = rag_answer(user_query)
print(f"Ответ модели:\n{final_answer}")

Сервер Ollama активен
Вопрос: Какие правила включает в себя политика в отношении обработки персональных данных в Озон Инвест?
------------------------------
Найденные чанки (первые 300 символов каждого)
Чанк 1:
ийской Федерации, если иное прямо не указано в Политике. 2. Принципы обработки персональных данных 2.1. Оператором разработаны и введены в действие внутренние документы и локальные нормативные акты, устанавливающие порядок обработки и защиты персональных данных, которые обеспечивают соответствие тре

Чанк 2:
поручения на обработку данных (с правом дальнейшей передачи третьим лицам (субобработчикам) в целях исполнения Договора); 9.5.2. Клиент обязуется соблюдать конфиденциальность ПДн и обеспечивать безопасность ПДн при их обработке, а также обеспечивает принятие необходимых правовых, организационных и т

Чанк 3:
соответствии с ним нормативных правовых актов, настоящей политики, внутренних документов и локальных актов по вопросам обработки персональных данных. 8.3. По запросу упо

Помимо изменения промпта мы увеличили топ с 3 до 5, что также положительно сказалось на результате работы модели.

## Улучшение RAG

Была идея с иерархией чанков, чтобы делить их на родительские и дочерние, которые ссылаются на родителей, но эта идея оказалась слишком затратной по мощностям. Мы разворачиваемся в коллабе, а здесь не хватит вычислительных мощностей чтобы покрыть просчет индексации новых чанков.

Поэтому было принято решение обойтись без иерархии чанков и совершенствовать алгоритм другими путями.

Увеличим размер чанка до 512, перекрытие будет 256. Сделаем разбиение по границам предложений. Реализуем гибридный поиск с RRF (Reciprocal Rank Fusion) вместо взвешенной суммы (alpha). Переранжирование будет через cross-encoder.


### Новая функция для чанков

In [ ]:
def split_into_chunks(text, chunk_size=512, overlap=256):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Ищем границу предложения для аккуратного деления
        if end < len(text):
            last_period = text.rfind('.', start, end)
            if last_period != -1 and last_period > start:
                end = last_period + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end
    return chunks

## Гибридный поиск с RRF (Reciprocal Rank Fusion)

Реализуем гибридный поиск с RRF, при этом выставим параметр k = 60.

In [ ]:
def hybrid_search_rrf(query, top_k=15, k_rrf=60):
    # Dense поиск (FAISS)
    q_emb = dense_model.encode([query])
    faiss.normalize_L2(q_emb)
    dense_scores, dense_indices = index.search(q_emb, top_k)
    # Ранги (чем меньше индекс, тем выше ранг)
    dense_ranks = {int(idx): rank+1 for rank, idx in enumerate(dense_indices[0])}

    # BM25 поиск
    bm25_scores = bm25.get_scores(query.split())
    bm25_top_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_ranks = {int(idx): rank+1 for rank, idx in enumerate(bm25_top_indices)}

    # RRF слияние
    rrf_scores = {}
    for idx, rank in dense_ranks.items():
        rrf_scores[idx] = 1 / (k_rrf + rank)
    for idx, rank in bm25_ranks.items():
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k_rrf + rank)

    # Сортируем по убыванию RRF скора
    sorted_indices = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i], reverse=True)
    # Возвращаем чанки с их RRF скорами (можно ограничить top_k)
    return [(all_chunks[idx], rrf_scores[idx]) for idx in sorted_indices[:top_k]]

Соответственно, теперь поменяем метод для поиска в функции всего цикла RAG.

In [ ]:
def rag_answer(query, model="llama3.2:3b"):
    initial_results = hybrid_search_rrf(query, top_k=15, k_rrf=60)
    best_chunks = rerank(query, initial_results, top_n=5)

    print("Найденные чанки (первые 300 символов каждого)")
    for i, chunk in enumerate(best_chunks):
      print(f"Чанк {i+1}:\n{chunk['text'][:300]}\n")

    if not best_chunks:
        return "Не найдено релевантных фрагментов."

    context_text = "\n\n".join([c["text"] for c in best_chunks])

    prompt = f"""Ты — ассистент, отвечающий строго на основе предоставленного контекста.
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."

**Контекст:**
{context_text}

**Вопрос:** {query}

**Ответ:**"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=90
        )
        if response.status_code == 200:
            return response.json()["response"].strip()
        else:
            return f"Ошибка API: {response.status_code}"
    except Exception as e:
        return f"Ошибка: {e}"


## Валидация

Напишем код для проверки корректности поиска модели. Будем проверять по 10 тестовым запросам. Для тестовых запросов сделаем пары вопрос-ответ.

In [ ]:
import time
import pandas as pd

# Тестовые запросы
test_queries = [
  {"id": "q1", "query": "В какой валюте и в какие сроки Клиент оплачивает услуги Экспедитора по договору перевозки WB Drive?",
   "expected": ["российских рублях", "14 календарных дней", "с даты получения поручения на оплату и УПД"]},

  {"id": "q2", "query": "Какова сумма предусмотренного оценочного резерва (provision) на судебные издержки (adverse costs order) в деле о патентном нарушении?",
   "expected": ["GBP 50,000", "GBP 150,000", "between 50,000 and 150,000"]},

  {"id": "q3", "query": "Какую сумму рекомендации по урегулированию (settlement) предлагает Legal Department по иску Dr. Rachel Williams?",
   "expected": ["GBP 85,000", "12 months' salary", "injury to feelings award"]},

  {"id": "q4", "query": "Каков срок действия Master Supply Agreement между Veracier UK и Smithfield Coatings plc?",
   "expected": ["three (3) years", "автоматическое продление", "one (1) year periods", "six (6) months' notice"]},

  {"id": "q5", "query": "Какое минимальное целевое значение p99 латентности запрашивает клиент Helio и какой вариант предлагает Redwood для его достижения?",
   "expected": ["p99 \u2264 150 ms", "Dedicated", "pinned GPU pool", "компромиссы (меньшая модель или агрессивное квантование)"]},

  {"id": "q6", "query": "Каким образом в режиме «Формы в закладках» можно открыть форму, расположенную на закладке, в отдельном окне?",
   "expected": ["Открыть в отдельном окне", "контекстного меню закладки"]},

  {"id": "q7", "query": "Какие пороговые значения пропускной способности (throughput) были согласованы для приемочного тестирования Aperture Finance?",
   "expected": ["15k tokens/sec", "12k tokens/sec", "initial go/no-go"]},

  {"id": "q8", "query": "Какая ставка роялти за внутригрупповые лицензии на ИС упоминается в документах как соответствующая бенчмаркингу?",
   "expected": ["2%", "от чистых продаж", "interquartile range 1.5% to 3.0%"]},

  {"id": "q9", "query": "В течение какого срока может быть предъявлен иск по патентному спору согласно закону Англии и Уэльса (Limitation Act 1980) и когда истекает этот срок в данном деле?",
   "expected": ["six (6) years", "15 March 2028", "Limitation Act 1980"]},

  {"id": "q10", "query": "Кто является конечным подписантом со стороны Globex в цепочке подписания DPA, согласно переписке?",
   "expected": ["James Li", "VP Legal", "final approver"]},
]

# Функция recall@k
def simple_recall(query, expected, k=3):
    # Поиск и переранжирование (используем уже существующие функции)
    candidates = hybrid_search_rrf(query, top_k=15, k_rrf=60)
    best_chunks = rerank(query, candidates, top_n=k)
    if not best_chunks:
        return 0.0, []
    retrieved_text = " ".join([ch["text"] for ch in best_chunks]).lower()
    found = sum(1 for frag in expected if frag.lower() in retrieved_text)
    return found / len(expected) if expected else 1.0, best_chunks

# Прогон
results = []
for q in test_queries:
    start = time.time()
    recall, chunks = simple_recall(q["query"], q["expected"], k=3)
    elapsed = time.time() - start
    results.append({"id": q["id"], "query": q["query"], "recall@3": recall, "time": elapsed})
    print(f"{q['id']}: recall@3={recall:.2f}, time={elapsed:.2f}s")
    if chunks:
        print(f"  Топ-1 чанк: {chunks[0]['text'][:500]}...\n")

df = pd.DataFrame(results)
print("\n=== ИТОГО ===")
print(f"Средний recall@3: {df['recall@3'].mean():.2f}")
df.to_csv("light_validation.csv", index=False)
print("Результаты сохранены в light_validation.csv")

q1: recall@3=0.00, time=3.60s
  Топ-1 чанк: слуги Стороны согласовывают и отражают в Заявке. Обязательства по оказанию услуги ПМ возникают у Экспедитора при приеме Заявки в Программе или выборе Водителем типа перевозки СЦ/РЦ - ПВЗ, ПВЗ-СЦ/РЦ. Представитель - лицо, подключенное к Системе, действующее от имени и в интересах или за счет Экспедитора, полномочия которого основаны на распоряжении Экспедитора, доверенности,...

q2: recall@3=0.00, time=3.08s
  Топ-1 чанк: Экспедитором третьего лица нарушений на территории Клиента/Грузополучателя/Грузоотправителя, Экспедитор уплачивает по требованию Клиента штраф в размере, установленном в таблице ниже, за каждый выявленный случай: Нарушение* Предельная сумма штрафа, руб. 10.9.1. пеший проход под шлагбаумом или проход в обход турникета внутрь Логистического Объекта (Склада/РЦ/СЦ/ иного) 10 000...

q3: recall@3=0.00, time=1.84s
  Топ-1 чанк: ве перенаправить требования третьих лиц Экспедитору или удержать сумму ущерба из вознаграждения Экспедито

Получился подозрительно низкий результат. Скорее всего это связано с тем, что  идет поиск строгого совпадения, то есть если слова одно и те же, но разное окончание, будет засчитан промах. Стоит исправить это. Добавим стеммер, удаляющий типичные окончания слов. В таком случае точность должна возрасти.

Также добавим LLM-судью на случай, если простая проверка ошиблась. Для судьи будем использовать также llama3.2:3b. Он будет оценивать ответ модели по соответствию контексту, то есть да/нет, которые мы будем переводить в 0-1.

In [ ]:
import re
import time
import pandas as pd
import requests

# Простой стеммер (удаление типичных окончаний)
def simple_stem(word):
    word = word.lower()
    # Удаляем окончания
    for suffix in ['ая', 'яя', 'ые', 'ие', 'ой', 'ей', 'ую', 'юю', 'ого', 'ему', 'ым', 'им', 'ом', 'ем',
                   'ая', 'яя', 'ие', 'ые', 'ое', 'а', 'я', 'о', 'е', 'и', 'ы', 'у', 'ю', 'ь', 'й']:
        if word.endswith(suffix):
            word = word[:-len(suffix)]
            break
    if len(word) < 3:
        return word
    return word

def normalize_text(text):
    # Извлекаем слова, приводим к стему
    words = re.findall(r'\b[а-яё]+\b', text.lower())
    stems = [simple_stem(w) for w in words]
    return set(stems)

def recall_fuzzy(query, expected, k=3):
    candidates = hybrid_search_rrf(query, top_k=50, k_rrf=30)
    best_chunks = rerank(query, candidates, top_n=k)
    if not best_chunks:
        return 0.0, []
    # Собираем все стемы из найденных чанков
    all_stems = set()
    for chunk in best_chunks:
        all_stems.update(normalize_text(chunk["text"]))
    # Проверяем каждую ожидаемую фразу
    found = 0
    for phrase in expected:
        phrase_stems = normalize_text(phrase)
        # Если все стемы фразы присутствуют в наборе – считаем фразу найденной
        if phrase_stems.issubset(all_stems):
            found += 1
    recall = found / len(expected) if expected else 1.0
    return recall, best_chunks

def generate_answer_from_chunks(query, chunks, model="llama3.2:3b"):
    if not chunks:
        return "Не найдено релевантных фрагментов."

    context_text = "\n\n".join([c["text"] for c in chunks])

    prompt = f"""Ты — ассистент, отвечающий строго на основе предоставленного контекста. "
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."

**Контекст:**
{context_text}

**Вопрос:** {query}

**Ответ:**"""
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=90
        )
        if response.status_code == 200:
            return response.json()["response"].strip()
        else:
            return f"Ошибка API: {response.status_code}"
    except Exception as e:
        return f"Ошибка: {e}"


# Тестовые запросы
test_queries = [
  {"id": "q1", "query": "В какой валюте и в какие сроки Клиент оплачивает услуги Экспедитора по договору перевозки WB Drive?",
   "expected": ["российских рублях", "14 календарных дней", "с даты получения поручения на оплату и УПД"]},

  {"id": "q2", "query": "Какова сумма предусмотренного оценочного резерва (provision) на судебные издержки (adverse costs order) в деле о патентном нарушении?",
   "expected": ["GBP 50,000", "GBP 150,000", "between 50,000 and 150,000"]},

  {"id": "q3", "query": "Какую сумму рекомендации по урегулированию (settlement) предлагает Legal Department по иску Dr. Rachel Williams?",
   "expected": ["GBP 85,000", "12 months' salary", "injury to feelings award"]},

  {"id": "q4", "query": "Каков срок действия Master Supply Agreement между Veracier UK и Smithfield Coatings plc?",
   "expected": ["three (3) years", "автоматическое продление", "one (1) year periods", "six (6) months' notice"]},

  {"id": "q5", "query": "Какое минимальное целевое значение p99 латентности запрашивает клиент Helio и какой вариант предлагает Redwood для его достижения?",
   "expected": ["p99 \u2264 150 ms", "Dedicated", "pinned GPU pool", "компромиссы (меньшая модель или агрессивное квантование)"]},

  {"id": "q6", "query": "Каким образом в режиме «Формы в закладках» можно открыть форму, расположенную на закладке, в отдельном окне?",
   "expected": ["Открыть в отдельном окне", "контекстного меню закладки"]},

  {"id": "q7", "query": "Какие пороговые значения пропускной способности (throughput) были согласованы для приемочного тестирования Aperture Finance?",
   "expected": ["15k tokens/sec", "12k tokens/sec", "initial go/no-go"]},

  {"id": "q8", "query": "Какая ставка роялти за внутригрупповые лицензии на ИС упоминается в документах как соответствующая бенчмаркингу?",
   "expected": ["2%", "от чистых продаж", "interquartile range 1.5% to 3.0%"]},

  {"id": "q9", "query": "В течение какого срока может быть предъявлен иск по патентному спору согласно закону Англии и Уэльса (Limitation Act 1980) и когда истекает этот срок в данном деле?",
   "expected": ["six (6) years", "15 March 2028", "Limitation Act 1980"]},

  {"id": "q10", "query": "Кто является конечным подписантом со стороны Globex в цепочке подписания DPA, согласно переписке?",
   "expected": ["James Li", "VP Legal", "final approver"]},

 ]


def diagnose_query(query, expected, k=5):
    candidates = hybrid_search_rrf(query, top_k=50, k_rrf=30)
    best_chunks = rerank(query, candidates, top_n=k)
    print(f"Запрос: {query}")
    print(f"Найдено чанков: {len(best_chunks)}")
    all_text = " ".join([ch["text"] for ch in best_chunks]).lower()
    for phrase in expected:
        found = phrase.lower() in all_text
        print(f"  {'+' if found else '-'} {phrase}")
    print("\nТоп-3 чанка (первые 150 символов):")
    for i, ch in enumerate(best_chunks[:3]):
        print(f"{i+1}. {ch['text'][:150]}...")
    print("-" * 50)

# для проблемных запросов
# diagnose_query(test_queries[2]["query"], test_queries[2]["expected"])

def faithfulness_judge(query, context_chunks, answer, model="llama3.2:3b", retries=2):
    if not context_chunks:
        return 0
    context = "\n\n".join([c["text"] for c in context_chunks[:3]])
    prompt = f"Вопрос: {query}\nКонтекст: {context[:1500]}\nОтвет: {answer}\nОтвет основан ТОЛЬКО на контексте? Ответь 'да' или 'нет'."
    for _ in range(retries):
        try:
            resp = requests.post("http://localhost:11434/api/generate",
                                 json={"model": model, "prompt": prompt, "stream": False,
                                       "options": {"temperature": 0.0}},
                                 timeout=60)
            result = resp.json()["response"].strip().lower()
            if "да" in result or result == "yes":
                return 1
            else:
                return 0
        except:
            time.sleep(2)
    return 0

# Прогон
results = []
for q in test_queries:
    start = time.time()
    recall, chunks_for_evaluation = recall_fuzzy(q["query"], q["expected"], k=3)
    answer = generate_answer_from_chunks(q["query"], chunks_for_evaluation)

    elapsed = time.time() - start

    faithful = faithfulness_judge(q["query"], chunks_for_evaluation, answer)

    results.append({"id": q["id"], "query": q["query"], "recall@3": recall, "faithfulness": faithful, "time": elapsed})
    print(f"{q['id']}: recall@3={recall:.2f}, faithful={faithful}, time={elapsed:.2f}s")
    if chunks_for_evaluation:
        print(f"  Топ-1 чанк: {chunks_for_evaluation[0]['text'][:200]}...\n")

df = pd.DataFrame(results)
print("\nИТОГО")
print(f"Средний recall@3: {df['recall@3'].mean():.2f}")
print(f"Средняя faithfulness: {df['faithfulness'].mean():.2f}")
print(f"Среднее время на ответ: {df['time'].mean():.2f}")

df.to_csv("light_validation.csv", index=False)



q1: recall@3=0.00, faithful=1, time=8.25s
  Топ-1 чанк: вия корректировки маршрутов и выездов 2.1. Клиент имеет право скорректировать точки погрузки и выгрузки, отличные от указанных в Техническом задании, при этом такие изменения допускаются в пределах ра...

q2: recall@3=1.00, faithful=1, time=9.53s
  Топ-1 чанк: дитора, а также привлеченных им третьих лиц (Экспедитора), предусмотренного Договором или иными гражданско-правовыми договорами, заключенными между Сторонами, в рамках которых Клиент обязан оплачивать...

q3: recall@3=1.00, faithful=0, time=8.49s
  Топ-1 чанк: ковая сигнализация при превышении порога по МЭД – установленный флажок программно активирует включение звуковой сигнализации при превышении установленных порогов по МЭД; Звуковая сигнализация при прев...

q4: recall@3=0.75, faithful=1, time=6.68s
  Топ-1 чанк: язательств в срок, установленный в Договоре, то этот срок соразмерно отодвигается на время действия такого обстоятельства. 5.12. В случае наступления обстоятельс

Теперь результат уже больше похож на правду. recall@3 = 0.82 вполне хороший показатель для модели. LLM-судья дал оценку 0.50. Можно предположить, что судья дал такую низку оценку из-за того, что работает с ошибками, поскольку является той же моделью с теми же мощностями.

Среднее время ответа составило 8.88 секунд, что вполне приемлемо для локальной модели.

В целом в сравнении с baseline улучшенная модель ведет себя лучше, и дает более правильные ответы на запросы. Периодически у нее выскакивают ошибки, в том числе из-за возможных галлюцинаций.
Врмея ответа увеличилось примерно с 5 секунд почти до 9, но зато возросло качество ответа. Был усовершенствован поиск: вместо взешенной суммы по alpha = 0.6 теперь используется RRF. Также мы использовали переранжирование с помощью кросс-енкодера, увеличили чанки.
Гиперпараметры:
- chunk_size = 512
- overlap = 256
- top_k = 50
- k_rrf = 30
- rerank_tor_n = 5
- model = llama3.2:3b

